# K5 · ResearchLens end-to-end test (Part 2 + Part 3) on a GPU
1. Install the app requirements (Qdrant, pdfplumber, Streamlit, transformers with Qwen3-VL).
2. Get two papers: **upload your own PDFs** to `/kaggle/working/papers/` (Kaggle: *Add Input → Upload*), otherwise two arXiv
   papers are downloaded as a demo (SciBERT and ViT).
3. **Process** each paper: parse → our 6 fine-tuned models → embeddings → Qdrant.
4. **Ask** questions, **overview**, **explain** a figure/formula, **compare**, **Gap Map**, **ideas** — each with an Evidence Trace.
5. Optional: start the **Streamlit app** with a temporary public link.

## 0. Setup (Kaggle)
Before running: right panel → **Accelerator: GPU T4 x2 (or P100)**, **Internet: On** (needs a phone-verified Kaggle account),
and **Add-ons → Secrets → add `HF_TOKEN`** (your Hugging Face write token) and tick it for this notebook.

Kaggle has no Google Drive: results go to `/kaggle/working/ResearchLens` **and** every model + its results are pushed to your
private Hugging Face account. The last cell also backs up all results to a private HF dataset.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
DRIVE_ROOT = "/kaggle/working/ResearchLens"          # results (Kaggle's saved output folder)
SCRATCH = "/tmp/rl"                                    # data, checkpoints, final models (large, not saved)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # one GPU, same setup as the Colab runs
for d in (DRIVE_ROOT, f"{SCRATCH}/content"):
    os.makedirs(d, exist_ok=True)
if not os.path.exists("/content"):                     # configs use /content/... paths (Colab layout)
    try:
        os.symlink(f"{SCRATCH}/content", "/content")
    except OSError as e:
        print("note: could not create /content link:", e)
os.makedirs("/content/data", exist_ok=True)

if not os.path.exists(f"{SCRATCH}/repo"):
    !git clone -q {REPO_URL} {SCRATCH}/repo
else:
    !git -C {SCRATCH}/repo pull -q
%cd {SCRATCH}/repo
!pip install -q -r requirements-colab.txt

os.environ.update({
    "RL_DRIVE_ROOT": DRIVE_ROOT, "RL_DATA_DIR": "/content/data",
    "RL_CKPT_DIR": f"{SCRATCH}/checkpoints", "RL_MODEL_DIR": f"{SCRATCH}/final_models",
    "TOKENIZERS_PARALLELISM": "false", "TQDM_DISABLE": "1",
})
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN found")
except Exception as e:
    print("WARNING: HF_TOKEN secret missing / not attached -> models will NOT be pushed to the Hub.", e)
!nvidia-smi --query-gpu=name,memory.total --format=csv

def run(module, config, run_name, seed, extra=""):
    # Train one (run, seed) unless it already finished; streams the log and stops on failure.
    import yaml
    tag = yaml.safe_load(open(config))["model_tag"]
    if os.path.exists(f"{DRIVE_ROOT}/results/{tag}/{run_name}/seed_{seed}/metrics.json"):
        print(f"skip {run_name} seed {seed} (finished)"); return
    if "--push_to_hub" in extra and not os.environ.get("HF_TOKEN"):
        extra = extra.replace("--push_to_hub", "")
    cmd = f"{sys.executable} -m {module} --config {config} --run_name {run_name} --seed {seed} {extra}"
    print(">>>", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    if p.wait() != 0:
        raise RuntimeError(f"Training failed: {run_name} seed {seed}. Last lines:\n" + "".join(tail))

In [ ]:
# the app uses BOTH T4 GPUs (Qwen3-VL is split over them); must be set before torch touches the GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

## 1. Install the app requirements

In [ ]:
!pip install -q -r requirements-app.txt
import transformers, qdrant_client; print("transformers", transformers.__version__)

## 2. Papers

In [ ]:
import glob, urllib.request
PAPERS = "/kaggle/working/papers"; os.makedirs(PAPERS, exist_ok=True)
pdfs = sorted(glob.glob(f"{PAPERS}/*.pdf") + glob.glob("/kaggle/input/**/*.pdf", recursive=True))[:2]
if not pdfs:
    for name, url in [("scibert.pdf", "https://arxiv.org/pdf/1903.10676"), ("vit.pdf", "https://arxiv.org/pdf/2010.11929")]:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ResearchLens course project)"})
        open(f"{PAPERS}/{name}", "wb").write(urllib.request.urlopen(req, timeout=60).read())
    pdfs = sorted(glob.glob(f"{PAPERS}/*.pdf"))
print(pdfs)

## 3. Process the papers (parse → our models → Qdrant)

In [ ]:
import json, time
from researchlens.pipeline.models import ModelBank
from researchlens.pipeline.index import client
from researchlens.pipeline.run import process_paper
WORK = "/kaggle/working/rl_papers"
bank = ModelBank()
qc = client(f"{WORK}/qdrant")
docs = [process_paper(p, WORK, bank, qc, backend="pdfplumber") for p in pdfs]
for d in docs:
    print("\n==", d["title"][:90])
    print("entities:", [(e["text"], e["type"]) for e in d["entities"][:15]])
    print("relations:", [(r["head"], r["relation"], r["tail"], r["conf"]) for r in d["relations"][:8]])
    for b in d["blocks"]:
        if b["kind"] in ("figure", "table"):
            print(f"  {b.get('label'):<10} p.{b['page']} type={b.get('figure_type')} ({b.get('figure_type_conf')}) "
                  f"sources={b.get('figure_type_sources')} mentioned_in={len(b.get('mentioned_in', []))}")
        if b["kind"] == "equation":
            print(f"  {b.get('label'):<10} p.{b['page']} TrOCR: {b.get('latex_trocr', '')[:80]}")

## 4. Ask questions (dynamic multimodal retrieval → Qwen3-VL → Evidence Trace)

In [ ]:
from researchlens.pipeline.answer import ask
ids = [d["paper_id"] for d in docs]
def show(r):
    print(r["answer"]); print("\n-- Evidence Trace --")
    for t in r["trace"]:
        print(f"  [{t['eid']}] p.{t['page']} {t.get('section') or ''} {t.get('label') or t['kind']} | {t['why']} | {t['snippet'][:100]}")
t0 = time.time()
for q in ["Which datasets are used for evaluation?", "What does the main results table show?",
          "What is the main limitation of the approach?"]:
    print("\n#####", q); show(ask(bank, q, ids[:1], qc=qc))
print(f"\n{time.time()-t0:.0f}s for 3 questions")

## 5. Overview + explain a figure / formula

In [ ]:
from researchlens.pipeline.analysis import overview, explain_item, compare, gap_map, ideas, load_doc
docs = [load_doc(d["out_dir"]) for d in docs]
ov = overview(bank, docs[0], qc=qc); print(ov["overview"])
labs = [b["label"] for b in docs[0]["blocks"] if b["kind"] in ("figure", "equation") and b.get("label")]
for lab in labs[:2]:
    print("\n#####", lab); r = explain_item(bank, docs[0], lab, qc=qc); print(r["answer"])

## 6. Compare, Gap Map, ideas (needs 2 papers)

In [ ]:
if len(docs) > 1:
    c = compare(bank, docs[0], docs[1], qc=qc); print(c["comparison"])
    g = gap_map(bank, docs[0], docs[1], qc=qc)
    print(json.dumps(g["gaps"], indent=1)[:3000])
    i = ideas(bank, [x for x in g["gaps"] if x["supported"]], docs[0], docs[1])
    print(json.dumps(i["ideas"], indent=1)[:3000])
    json.dump({"overview": ov["overview"], "comparison": c["comparison"], "gaps": g["gaps"], "ideas": i["ideas"]},
              open("/kaggle/working/k5_outputs.json", "w"), indent=1)

## 7. (Optional) Run the Streamlit app with a temporary public link
Starts the app on this GPU and prints a `trycloudflare.com` link (works while this notebook session runs).

In [ ]:
import subprocess, re, time
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
os.environ["RL_WORK_DIR"] = "/kaggle/working/rl_app"
# free the notebook's GPU memory so the app can load the models itself
import gc, torch
for name in ("bank", "ov", "c", "g", "i", "r"):
    globals().pop(name, None)
gc.collect(); torch.cuda.empty_cache()
app = subprocess.Popen("streamlit run app/streamlit_app.py --server.port 8501 --server.headless true", shell=True,
                       stdout=open("/tmp/streamlit.log", "w"), stderr=subprocess.STDOUT)
tun = subprocess.Popen("/tmp/cloudflared tunnel --url http://localhost:8501 --no-autoupdate", shell=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("ResearchLens app:", m.group(0)); break